In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 유니터리 연산 합성하기

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
```
</details>
유니터리 연산은 양자 시스템에 대한 노름 보존 변환을 나타냅니다.
$n$ Qubit에 대해 이 변환은 $2^n \times 2^n$ 차원의 복소 행렬 $U$로 표현되며, 이 행렬의 에르미트 켤레가 역행렬과 같습니다. 즉, $U^\dagger U = \mathbb{1}$입니다.

특정 유니터리 연산을 양자 Gate 집합으로 합성하는 것은 양자 알고리즘의 설계 및 적용, 또는 양자 Circuit 컴파일 등에 활용되는 기본적인 작업입니다.

Clifford Gate로 구성된 유니터리나 텐서 곱 구조를 가진 유니터리 등 특정 클래스의 유니터리에 대해서는 효율적인 합성이 가능하지만, 대부분의 유니터리는 이러한 범주에 속하지 않습니다.
일반적인 유니터리 행렬의 경우, 합성은 Qubit 수가 증가할수록 계산 비용이 지수적으로 늘어나는 복잡한 작업입니다.
따라서 구현하려는 유니터리에 대한 효율적인 분해 방법을 알고 있다면, 그 방법이 일반적인 합성보다 더 나을 가능성이 높습니다.

> **Note:** 분해 방법을 알 수 없는 경우, Qiskit SDK가 이를 찾을 수 있는 도구를 제공합니다.
>     단, 이 경우 일반적으로 깊은 Circuit이 생성되어 잡음이 있는 양자 컴퓨터에서 실행하기 적합하지 않을 수 있습니다.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

U = 0.5 * np.array(
    [[1, 1, 1, 1], [-1, 1, -1, 1], [-1, -1, 1, 1], [-1, 1, 1, -1]]
)

circuit = QuantumCircuit(2)
circuit.unitary(U, circuit.qubits)

## Re-synthesis for circuit optimization

Sometimes it is beneficial to re-synthesize a long series of single- and two-qubit gates, if the length can be reduced. For example, the following circuit uses three two-qubit gates.

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

qreg_q = QuantumRegister(2, "q")
creg_c = ClassicalRegister(4, "c")
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.h(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.sx(qreg_q[1])
circuit.cz(qreg_q[0], qreg_q[1])
circuit.x(qreg_q[1])
circuit.x(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.h(qreg_q[0])
circuit.draw("mpl")

<Image src="../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg" alt="Output of the previous code cell" />

## Circuit 최적화를 위한 재합성
경우에 따라 단일 Qubit 및 2-Qubit Gate의 긴 시퀀스를 재합성하면 길이를 줄일 수 있어 유익할 수 있습니다. 예를 들어 다음 Circuit은 3개의 2-Qubit Gate를 사용합니다.

In [3]:
from qiskit.quantum_info import Operator

# compute unitary matrix of circuit
U = Operator(circuit)

# re-synthesize
better_circuit = QuantumCircuit(2)
better_circuit.unitary(U, range(2))
better_circuit.decompose().draw()

global phase: 6.2071
      ┌───────────────┐         ┌────────────────┐ 
q_0: ─┤ U(π/2,π/2,-π) ├────■────┤ U(π/2,-π,-π/2) ├─
     ┌┴───────────────┴─┐┌─┴─┐┌─┴────────────────┴┐
q_1: ┤ U(1.7229,π/2,-π) ├┤ X ├┤ U(π/2,0.15207,-π) ├
     └──────────────────┘└───┘└───────────────────┘

![Output of the previous code cell](../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg)

그러나 다음 코드로 재합성하면 단일 CX Gate만 필요합니다. (여기서는 `QuantumCircuit.decompose()` 메서드를 사용하여 유니터리 재합성에 사용된 Gate를 더 잘 시각화합니다.)